In [1]:
from datasets import load_dataset
from typing import List, Tuple

class TextSourceClassifier:
    def __init__(
        self,
        dataset_name: str = "NabeelShar/ai_and_human_text",
        max_texts: int = 200,
        ai_model_preference: List[str] = None,
    ):
        """
        Initialize the classifier with a dataset from Hugging Face.

        Args:
            dataset_name (str): Name of the dataset on Hugging Face.
                               Default: "Ateeqq/AI-and-Human-Generated-Text".
            max_texts (int): Maximum number of texts to retrieve.
                            Default: 1000.
            ai_model_preference (List[str]): List of AI models to prefer (e.g., ["gpt2", "llama"]).
                            Default: None (no preference).
        """
        self.dataset_name = dataset_name
        self.max_texts = max_texts
        self.ai_model_preference = ai_model_preference or ["gpt2", "llama"]
        self.dataset = None

    def load_data(self) -> None:
        """Load the dataset from Hugging Face."""
        try:
            self.dataset = load_dataset(self.dataset_name, split="train")
            print(f"Dataset '{self.dataset_name}' loaded successfully.")
        except Exception as e:
            print(f"Failed to load dataset: {e}")
            raise

    def filter_and_get_texts_with_labels(self) -> List[Tuple[str, int]]:
        """
        Filter the dataset to include only texts from older AI models or human-written,
        and return a list of texts with their labels (0 for human, 1 for AI).

        Returns:
            List[Tuple[str, int]]: A list of tuples where each tuple contains a text and its label.
        """
        if self.dataset is None:
            self.load_data()

        texts = []
        y_true = []
        ai_count = 0
        human_count = 0
        max_ai = self.max_texts // 2
        max_human = self.max_texts // 2

        for example in self.dataset:
            text = example.get("text", "")
            label = example.get("generated", None)
            #model_info = example.get("model_info", None)  # Assuming a column like "model_info" exists

            if not text or label is None:
                continue

            # Prefer older AI models (e.g., GPT-2, Llama)
            if label == 1 and ai_count < max_ai:
                #if model_info and any(model in model_info.lower() for model in self.ai_model_preference):
                if len(text)<1000 and len(text)>100:
                    texts.append(text)
                    y_true.append(label)
                    ai_count += 1
            elif label == 0 and human_count < max_human:
                if len(text)<1000 and len(text)>100:
                    texts.append(text)
                    y_true.append(label)
                    human_count += 1

            # Stop if we have enough texts
            if ai_count >= max_ai and human_count >= max_human:
                break

        #print(model_info)
        return texts,y_true



In [2]:
datamodel = TextSourceClassifier()
datamodel.load_data()
texts,y_true = datamodel.filter_and_get_texts_with_labels()

Dataset 'NabeelShar/ai_and_human_text' loaded successfully.


In [ ]:
def read_lines_as_lists(file_path):
    perturbations = []
    with open(file_path, 'r') as f:
        for line in f:
            # Remove whitespace and newline characters
            stripped_line = line.strip()
            # Convert the string representation of a list into an actual list
            if stripped_line:  # Check if the line is not empty
                list_data = eval(stripped_line)
                perturbations.append(list_data)
    return perturbations

# Generate perturbations with perturbation model and save in txt file
file_path = "pregenerated_perturbations.txt"
perturbations = read_lines_as_lists(file_path)


In [ ]:
from __future__ import annotations

import numpy as np
import torch

from transformers import T5ForConditionalGeneration,T5Tokenizer

from detector import BayesianDetector
from af import build_af


if __name__ == "__main__":
    device = "cpu"

    num_perturbations = len(perturbations[0])
    N = len(texts)
    budget = 15
    seed = 0
    t5model = T5ForConditionalGeneration.from_pretrained('Vamsi/T5_Paraphrase_Paws')
    t5tokenizer = T5Tokenizer.from_pretrained('t5-base')

    detector = BayesianDetector(
        acquisition_fn_builder=lambda m: build_af(m, aqn_name="NIPV",N=N,seed=seed),
        t5model=t5model,
        t5tokenizer=t5tokenizer,
        device=device,
        random_sampling = False,
        seed = seed
    )

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
x1 = []
for idx,text in enumerate(texts):
        score = detector.detect(text,num_perturbations,budget,perturbations[idx])
        x1.append(score)
        
np.save("Results/1_NIPV_seed0",np.asarray(x1))
del x1

AssertionError: Expected the output shape to match either the t-batch shape of X, or the `model.batch_shape` in the case of acquisition functions using batch models; but got output with shape torch.Size([5, 201]) for X with shape torch.Size([201, 1, 1]).